# TabML Kaggle Setup Guide

This notebook demonstrates how to use TabML in Kaggle competitions using the ZIP archive method.

## Prerequisites

1. Create a ZIP archive of your TabML repository locally:
   ```bash
   git archive --format=zip --output=tabml.zip HEAD
   ```

2. Upload the ZIP file to Kaggle as a dataset:
   - Go to https://www.kaggle.com/datasets
   - Click "New Dataset"
   - Upload `tabml.zip`
   - Name it "tabml-repo" (or any name you prefer)
   - Set visibility to "Private" if your repo is private
   - **Note:** Kaggle automatically unzips the archive

3. Add the dataset to your Kaggle notebook:
   - In your notebook, click "Add Data"
   - Search for your dataset and add it
   - Files will be available at `/kaggle/input/your-dataset-name/`

## Step 1: Setup TabML (Simple Path Method)

In [ ]:
import sys
import os

# Kaggle automatically unzips the archive
# Your files are available at /kaggle/input/your-dataset-name/
# Update this path based on your dataset name
TABML_PATH = '/kaggle/input/tabml-repo'

# Check what's available in the dataset
print("Contents of TabML dataset:")
!ls -la {TABML_PATH}

# First install required dependencies
print("\nInstalling TabML dependencies...")
!pip install -q loguru optuna

# Add TabML to Python path
sys.path.insert(0, TABML_PATH)

# Import and verify
import tabml
print(f"\n✓ TabML version {tabml.__version__} loaded successfully!")

# Check available modules
from tabml import TabularPipeline, XGBoostModel, LightGBMModel
print("✓ Core modules imported successfully")

## Alternative: Method 2 - Proper Package Installation

Use this method if you need proper dependency management:

In [ ]:
# Alternative installation method with pip
# This ensures all dependencies are properly installed

# Copy the package to working directory first (since /kaggle/input is read-only)
!cp -r /kaggle/input/tabml-repo /kaggle/working/tabml

# Install with pip
!pip install -e /kaggle/working/tabml -q

# Verify installation
import tabml
print(f"✓ TabML {tabml.__version__} installed with pip")

## Step 2: Install Optional Dependencies

In [ ]:
# Install TabNet for neural network models (optional)
!pip install pytorch-tabnet -q

# Verify GPU availability (if using TabNet)
try:
    import torch
    if torch.cuda.is_available():
        print(f"✓ GPU available: {torch.cuda.get_device_name(0)}")
    else:
        print("✗ GPU not available, will use CPU")
except ImportError:
    print("PyTorch not installed")

# Check available models
from tabml import XGBoostModel, LightGBMModel, CatBoostModel
print("\n✓ Tree-based models available")

try:
    from tabml import TabNetModel
    print("✓ TabNet model available")
except ImportError:
    print("✗ TabNet not available (install pytorch-tabnet)")

## Step 3: Quick Test with Sample Data

In [ ]:
# Create sample data to test the installation
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification

# Generate sample data
X, y = make_classification(
    n_samples=1000,
    n_features=20,
    n_informative=15,
    n_redundant=5,
    random_state=42
)

# Create DataFrames
feature_cols = [f'feature_{i}' for i in range(20)]
train_df = pd.DataFrame(X, columns=feature_cols)
train_df['target'] = y

# Split into train and test
train_data, test_data = train_test_split(train_df, test_size=0.2, random_state=42)
test_data = test_data.drop('target', axis=1)

print(f"Train shape: {train_data.shape}")
print(f"Test shape: {test_data.shape}")

## Step 4: Use TabML Pipeline

In [ ]:
# Method 1: Direct model usage (without pipeline)
from tabml import XGBoostModel, LightGBMModel
from sklearn.model_selection import train_test_split

# Prepare data (using train_data from Step 3)
X = train_data.drop('target', axis=1)
y = train_data['target']
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Train XGBoost model
print("Training XGBoost model...")
xgb_model = XGBoostModel(params={'n_estimators': 100, 'max_depth': 5})
xgb_model.fit(X_train, y_train, X_val, y_val)

# Make predictions
predictions = xgb_model.predict(test_data)
print(f"\n✓ Generated {len(predictions)} predictions")
print(f"Prediction shape: {predictions.shape}")
print(f"Sample predictions: {predictions[:5]}")

# Method 2: Save DataFrames to files then use pipeline
# (This is the recommended approach for real competitions)
import os
os.makedirs('/kaggle/working/temp_data', exist_ok=True)

# Save to CSV files
train_data.to_csv('/kaggle/working/temp_data/train.csv', index=False)
test_data.to_csv('/kaggle/working/temp_data/test.csv', index=False)

# Now use TabularPipeline
from tabml import TabularPipeline

pipeline = TabularPipeline(
    data_dir='/kaggle/working/temp_data',
    task_type='classification'
)

# Load and process data
pipeline.load_data(
    train_file='train.csv',
    test_file='test.csv',
    target_column='target'
)

# Engineer features
pipeline.engineer_features()

# Train models (quick test)
pipeline.train_models(
    model_types=['xgboost'],
    optimize_hyperparams=False
)

# Get predictions
pipeline_predictions = pipeline.predict()
print(f"\n✓ Pipeline generated {len(pipeline_predictions)} predictions")

## Step 5: Competition Workflow Example

In [ ]:
# Example workflow for a Kaggle competition
# This example shows a complete workflow with feature engineering and ensemble

# Assume your competition data is at:
COMPETITION_DATA_PATH = '/kaggle/input/your-competition-name'

# Uncomment and run the following code for your actual competition:
"""
from tabml import TabularPipeline

# Initialize pipeline for competition
pipeline = TabularPipeline(
    data_dir=COMPETITION_DATA_PATH,
    task_type='classification',  # or 'regression'
    random_state=42
)

# Load competition data
pipeline.load_data(
    train_file='train.csv',
    test_file='test.csv',
    target_column='target',  # Update with actual target column
    id_column='id'  # Update with actual ID column
)

# Run full pipeline with ensemble
submission = pipeline.run_full_pipeline(
    model_types=['xgboost', 'lightgbm', 'catboost'],
    optimize_hyperparams=True,
    ensemble_method='oof',
    cv_folds=5
)

# Save submission
submission.to_csv('submission.csv', index=False)
print("✓ Submission saved to submission.csv")
"""

# Alternative: More detailed control over the pipeline
"""
from tabml import (
    DataLoader, FeatureEngineer, AdvancedFeatureEngineer,
    ModelTrainer, OOFEnsemble, CrossValidator
)

# Load data
loader = DataLoader(COMPETITION_DATA_PATH)
train_df, test_df = loader.load_data(
    train_file='train.csv',
    test_file='test.csv',
    target_column='target'
)

# Feature engineering
engineer = FeatureEngineer()
X_train = engineer.fit_transform(train_df.drop('target', axis=1))
X_test = engineer.transform(test_df)
y_train = train_df['target']

# Advanced features (optional)
adv_engineer = AdvancedFeatureEngineer()
X_train = adv_engineer.create_polynomial_features(X_train, degree=2)
X_test = adv_engineer.create_polynomial_features(X_test, degree=2)

# Train models with cross-validation
trainer = ModelTrainer(task_type='classification')
models = trainer.train_all_models(
    X_train, y_train,
    model_types=['xgboost', 'lightgbm', 'catboost'],
    optimize_hyperparams=True
)

# Create ensemble
ensemble = OOFEnsemble(task_type='classification')
oof_preds = ensemble.get_oof_predictions(models, X_train, y_train, n_folds=5)
weights = ensemble.optimize_weights(oof_preds, y_train, method='optuna')

# Generate final predictions
test_preds = ensemble.get_test_predictions(models, X_test)
final_predictions = ensemble.weighted_average(test_preds, weights)

# Create submission
submission = pd.DataFrame({
    'id': test_df['id'],
    'target': final_predictions
})
submission.to_csv('submission.csv', index=False)
"""

print("Update the code above with your competition-specific paths and parameters")

## Advanced: Using OOF Ensemble

In [ ]:
from tabml import OOFEnsemble, XGBoostModel, LightGBMModel, CatBoostModel

# Using the sample data from above
X_train = train_data.drop('target', axis=1)
y_train = train_data['target']
X_test = test_data

# Create multiple models
models = [
    XGBoostModel(params={'n_estimators': 100, 'max_depth': 5}),
    LightGBMModel(params={'n_estimators': 100, 'num_leaves': 31}),
    CatBoostModel(params={'iterations': 100, 'depth': 5, 'verbose': False})
]

# Create OOF ensemble
ensemble = OOFEnsemble(task_type='classification')

# Get OOF predictions
print("Generating OOF predictions...")
oof_preds = ensemble.get_oof_predictions(
    models, X_train, y_train, 
    n_folds=3,  # Use fewer folds for quick demo
    random_state=42
)

# Optimize weights
print("\nOptimizing ensemble weights...")
weights = ensemble.optimize_weights(
    oof_preds, y_train, 
    method='scipy'  # Quick optimization
)
print(f"Optimized weights: {weights}")

# Train models on full data and get test predictions
print("\nTraining on full data...")
for model in models:
    model.fit(X_train, y_train)

test_preds = ensemble.get_test_predictions(models, X_test)
final_predictions = ensemble.weighted_average(test_preds, weights)

print(f"\n✓ Final ensemble predictions shape: {final_predictions.shape}")
print(f"Sample predictions: {final_predictions[:5]}")

## Tips for Kaggle Competitions

1. **Update Dataset Version**: When you update your local TabML code, create a new version of your Kaggle dataset:
   - Create new ZIP: `git archive --format=zip --output=tabml.zip HEAD`
   - Upload as new version to your existing Kaggle dataset
   - Your notebook will automatically use the latest version

2. **Offline Mode**: The ZIP method works in Kaggle's offline submission environment

3. **GPU Usage**: Enable GPU in notebook settings if using TabNet models

4. **Memory Management**: For large datasets, use:
   ```python
   pipeline.load_data(sample_frac=0.1)  # For testing
   ```

5. **Custom Features**: Add competition-specific features:
   ```python
   from tabml import AdvancedFeatureEngineer
   engineer = AdvancedFeatureEngineer()
   df = engineer.create_interaction_features(df, cols=['col1', 'col2'])
   ```

6. **Submission Format**: Always check the submission format:
   ```python
   sample_submission = pd.read_csv('/kaggle/input/competition/sample_submission.csv')
   submission = pd.DataFrame({
       'id': test_ids,
       'target': predictions
   })
   ```

## Troubleshooting

### Common Issues and Solutions

1. **ImportError: No module named 'tabml'**
   - Make sure the ZIP was extracted correctly
   - Check that pip install command ran successfully

2. **TabNet not available**
   - Install with: `!pip install pytorch-tabnet`

3. **Out of Memory**
   - Reduce batch size in model params
   - Use sample_frac when loading data
   - Select fewer features

4. **Slow training**
   - Enable GPU in notebook settings
   - Reduce n_estimators/iterations
   - Use fewer folds in cross-validation

5. **Version conflicts**
   - Kaggle environments have pre-installed versions
   - TabML is designed to work with standard Kaggle environment